In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.getcwd())

import torch
from core.utils.config import load_config, print_config
from core.data.datamodule import PretrainDataModule
from core.training import setup_device, create_kde_sampler
from core.training.pretrain import setup_pretraining, train
from core.model.bobert import BobertForPretraining

In [ ]:

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {setup_device()}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BoBERT Configuration")

In [ ]:
sampler_fn = lambda stars: create_kde_sampler(
    stars,
    bandwidth=config['pretraining']['sampling']['kde_bandwidth'],
    num_bins=config['pretraining']['sampling'].get('num_bins', 100),
    strength=config['pretraining']['sampling'].get('strength', 0.1),
)

datamodule = PretrainDataModule(config, sampler_fn=sampler_fn)
datamodule.setup()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
model = BobertForPretraining.from_config(config, device)

summary = model.bert.get_summary()
print(f"\n--- BERT Encoder Information ---")
print(f"Total Parameters: {summary['trainable_parameters'] / 1e6:.2f}M")
print(f"Model Dimension: {model.bert.d_model}")
print(f"Number of Heads: {model.bert.n_heads}")
print(f"Number of Layers: {model.bert.n_layers}")

In [ ]:
module, trainer = setup_pretraining(config, datamodule.normalizer, model)

print(f"\nPretraining setup complete.")
print(f"Total epochs: {config['pretraining']['num_epochs']}")
print(f"Training samples: {len(datamodule.train_data)}")
print(f"Validation samples: {len(datamodule.val_data)}")

In [ ]:
train(module, trainer, datamodule)

print("\nBoBERT training completed!")